# Spark Architecture — The Cast of Characters

## Architecture Overview

```text
┌──────────────────────────────────────────────────────┐
│                    DRIVER NODE                       │
│                                                      │
│  - Runs your notebook code                           │
│  - Holds the SparkSession                            │
│  - Plans the work (DAG)                              │
│  - Schedules tasks                                   │
│  - Collects results                                  │
└────────────────────────┬─────────────────────────────┘
                         │
          ┌──────────────┼──────────────┐
          ↓              ↓              ↓
    ┌──────────┐   ┌──────────┐   ┌──────────┐
    │ Executor │   │ Executor │   │ Executor │
    │    1     │   │    2     │   │    3     │
    │          │   │          │   │          │
    │ - Tasks  │   │ - Tasks  │   │ - Tasks  │
    │ - Cache  │   │ - Cache  │   │ - Cache  │
    └──────────┘   └──────────┘   └──────────┘
```

## The Roles

| Component       | Job                                                                                  |
| --------------- | ------------------------------------------------------------------------------------ |
| Driver          | Brain — plans work, schedules tasks, and coordinates execution                       |
| Executors       | Muscle — run tasks and can cache data in memory                                      |
| Cluster Manager | Allocates resources and launches executors. Databricks manages much of this for you. |
| SparkSession    | Your entry point for Spark operations, such as `spark.read` and `spark.sql`          |

# The Three Levels of Work: Job, Stage, Task

This is one of the most important Spark concepts. Understanding it helps you read logs and optimize performance.


```
Action creates a Job → shuffle boundaries split the job into Stages → each stage runs one Task per partition.

1. Job
Spark is lazy. Transformations (filter, select, join) only build a plan.
Every action (count(), show(), write, collect()) triggers one job.
3 actions in a notebook means at least 3 jobs. 
(Sometimes one action makes extra small jobs, e.g. schema inference or a broadcast.)

2. Stage
Spark splits a job into stages wherever a shuffle is needed, meaning data must move between executors.
Narrow transformations (filter, select, withColumn, map) need no shuffle, so they stay in the same stage.
Wide transformations (groupBy, join, distinct, orderBy, repartition) need a shuffle, so they start a new stage.
Rule: number of stages ≈ number of shuffles + 1.

3. Task
1 task = 1 partition = 1 CPU core, the smallest unit of work.
A stage reading 80 partitions runs 80 tasks. After a shuffle, the next stage runs 200 tasks by default (spark.sql.shuffle.partitions), unless AQE merges them.

```

## Visual Example

```python
df = spark.read.parquet("/data/sales")        # Lazy read
filtered = df.filter("amount > 100")          # Transformation
grouped = filtered.groupBy("region").sum()    # Transformation
grouped.show()                                # Action
```

What happens when `show()` runs:

```text
Job 1
 |
 ├── Stage 1: Read + Filter + Partial Aggregation
 │
 │              SHUFFLE
 │                 ↓
 │
 └── Stage 2: Final Aggregation + Show
```

* **Stage 1:** Read data, filter rows, and perform partial aggregation where possible.
* **Shuffle:** Redistribute data so that rows for the same region can be grouped together.
* **Stage 2:** Finish the aggregation and return the requested results.

The exact number of stages and jobs depends on the physical execution plan.

##DAG
```
Directed Acyclic Graph. It’s Spark’s plan of all the steps needed to get your result.

Directed: steps flow one way (read, then filter, then join, then write).
Acyclic: no loops, so a step never goes back to an earlier one.
Graph: steps are boxes, and the arrows show which step depends on which.
```

##RDD ,DataFrame,DataSet
```
1. RDD (Resilient Distributed Dataset)
The oldest, lowest-level API. A distributed collection of objects that Spark doesn’t understand the inside of.
No schema, so Spark can’t optimize it (no Catalyst, no Tungsten).
You write how to do it (map, filter, reduceByKey with lambdas).
Use it only for very custom, low-level logic or unstructured data. Rare today.
2. DataFrame
Data organized into named columns, like a SQL table or a Pandas table but distributed.
Has a schema, so the Catalyst optimizer and Tungsten engine make it much faster than RDDs.
You write what you want (select, groupBy, SQL), and Spark decides how.
This is what you use in PySpark and Databricks, 99% of the time.
3. Dataset
A DataFrame plus type safety: errors like a wrong column type are caught when you compile, not at runtime.
Only in Scala and Java. Python has no Dataset API because Python isn’t statically typed.
In Scala, a DataFrame is just Dataset[Row]
```



# What Is a Shuffle?

A Shuffle redistributes data across partitions, often across executors.

## Operations That May Require a Shuffle

* `groupBy()`
* `join()`
* `distinct()`
* `repartition()`
* `orderBy()`
* `sortByKey()` in the RDD API

Whether a shuffle occurs depends on the operation, partitioning, join strategy, and execution plan.

## Why Is It Expensive?

* Data moves across the network.
* Intermediate data may be written to disk.
* Memory pressure can increase.
* Data skew can cause some tasks to process much more data than others.


# Partitions — The Unit of Parallelism

A Partition is a chunk of data that a task processes.

More partitions can increase parallelism, but too many partitions introduce scheduling and processing overhead.

## How Many Partitions?

Spark splits data into chunks called partitions. 1 partition = 1 task = processed by 1 CPU core. So partitions decide how much work runs in parallel.
```
The 2 rules interviewers expect
Size rule: about 128 MB per partition (100 to 200 MB is fine).
→ Partitions ≈ data size ÷ 128 MB.
Example: 10 GB → 10,240 MB ÷ 128 ≈ 80 partitions.
Cores rule: 2 to 4× the total cores, so every core stays busy and fast tasks can pick up slack.
Example: 5 executors × 4 cores = 20 cores → 40 to 80 partitions.
```

## The Classic Mistake

The default value of `spark.sql.shuffle.partitions` is often 200 in standard Spark configurations, although Databricks settings and runtime versions may differ.

* **Small data:** 1 GB split into 200 tiny tasks may create unnecessary overhead.
* **Huge data:** 1 TB split into only 200 partitions may produce oversized tasks and memory pressure.

Tune partitioning based on actual workload measurements. Adaptive Query Execution can also adjust shuffle partitioning automatically.

#Broadcast joins

A Broadcast Join is a Spark optimization used when one table is small enough to fit in memory on each executor and the other table is large.
Spark sends a copy of the small table to each executor so that it can join the data locally, without shuffling the large table's data across the cluster.

**Why is a broadcast join required?**
The main reason is to avoid an expensive shuffle of the large table.

```
from pyspark.sql.functions import broadcast

result = sales.join(
    broadcast(products),
    on="product_id",
    how="inner"
)
```

# Narrow vs. Wide Transformations

Transformations are commonly classified by whether they require data redistribution.

## Narrow Transformations — Usually No Shuffle

Each input partition contributes to a limited number of output partitions without redistributing its data across the cluster.

Examples:

* `filter()`
* `select()`
* `map()`
* `withColumn()`

`union()` generally does not require a shuffle by itself, although later operations may.

These transformations are often faster because they avoid network redistribution.

## Wide Transformations — Often Require a Shuffle

Data from multiple input partitions may need to move into different output partitions.

Examples:

* `groupBy()`
* `distinct()`
* `repartition()`
* `orderBy()`
* Many joins

Some joins can avoid a shuffle, such as broadcast joins or joins that can use existing compatible partitioning.

**Rule:** Minimize unnecessary wide transformations. Each shuffle can become a performance bottleneck.

**1. What happens when an executor fails?**
“Spark reschedules the failed tasks on other executors and rebuilds lost partitions from lineage, so the job keeps running. Only a driver failure kills the app.”

**2. What is lineage, and how does Spark recover lost partitions?**
Lineage = the recipe of how each DataFrame was built: read this file, then filter, then join, then aggregate. Spark keeps this recipe (the DAG), not copies of the data.
When a partition is lost, Spark replays only the steps needed for that one partition, starting from the source or the last saved point. Other partitions are untouched.
That’s the “Resilient” in RDD.
Narrow steps (filter, select) recover cheaply, since one lost partition needs one parent partition. Wide steps (after a shuffle) may need data from many parent partitions, which is costlier. Spark reuses shuffle files if they still exist.
Very long lineage (loops, many iterations) makes recovery slow. Fix it with checkpoint(), which saves data to reliable storage and cuts the lineage. In Databricks, writing to a Delta table does the same.

**3. collect() vs show() vs take()**
collect() -	Brings ALL rows to the driver	
show(n) -	Prints the first n rows (default 20) 
take(n) - Brings the first n rows to the driver	

# Reading the Spark UI

When a Spark job runs, Databricks provides access to the Spark UI, which helps you investigate performance.

## The Four Useful Tabs

| Tab             | What It Shows                                            |
| --------------- | -------------------------------------------------------- |
| Jobs            | Jobs and their associated stages                         |
| Stages          | Task-level metrics, shuffle activity, and execution time |
| Storage         | Cached and persisted RDDs and DataFrames                 |
| SQL / DataFrame | SQL queries, execution plans, and related metrics        |

Tab names and available details can vary by Databricks Runtime version.

## What to Look For

In the Stages tab, investigate:

* **Shuffle Read/Write:** How much data was redistributed.
* **Task Duration:** Compare minimum, median, and maximum durations.
* **Data Skew:** Check whether some tasks process significantly more data than others.
* **Spill:** Check whether intermediate data spills from memory to disk.

## Red Flags

* Very high shuffle write: Large intermediate data volumes or unnecessary shuffles may be present.
* Maximum task duration much higher than the median: Possible data skew, uneven input sizes, or straggler tasks.
* High spill: Tasks may be using more memory than available, or processing large intermediate data.

# The Three Common Bottlenecks

## 1. Too Many Shuffles

**Symptom:** Long job execution times and high shuffle read/write.

**Possible fixes:**

* Remove unnecessary `groupBy()`, joins, or repartitioning.
* Filter data before expensive operations.
* Check whether a broadcast join is appropriate.
* Reuse suitable partitioning where possible.

## 2. Data Skew

**Symptom:** One task takes much longer than the others.

**Possible fixes:**

* Use Adaptive Query Execution (AQE).
* Consider salting heavily skewed keys.
* Use broadcast joins when one side is sufficiently small.
* Investigate unusually frequent key values.

## 3. Too Many or Too Few Partitions

**Symptoms:**

* Too many: Very short tasks with high scheduling overhead.
* Too few: Underused CPU resources, oversized tasks, or memory problems.

**Possible fixes:**

* Tune `spark.sql.shuffle.partitions`.
* Use `repartition()` to increase or redistribute partitions.
* Use `coalesce()` to reduce partitions when appropriate.
* Allow AQE to adjust shuffle partitions when enabled.

# Adaptive Query Execution (AQE)

AQE is Spark's runtime optimization feature. It uses statistics collected during execution to improve the physical plan.

In Databricks, AQE is generally enabled by default for supported runtimes and configurations.

It can automatically:

1. Coalesce small shuffle partitions.
2. Change some sort-merge joins to broadcast joins when runtime statistics show that one side is small enough.
3. Handle skewed shuffle partitions by splitting oversized partitions.

AQE reduces the need for manual tuning, but it does not eliminate every performance issue.

You can check the relevant configuration:

```python
spark.conf.get("spark.sql.adaptive.enabled")
```

In Databricks, you may also encounter:

```python
spark.conf.get("spark.databricks.optimizer.adaptive.enabled")
```

The applicable setting depends on the runtime and configuration. Prefer checking the actual execution plan and effective settings rather than assuming a particular configuration name controls every AQE feature.

# Real-World Scenario: Investigating Data Skew

Problem: A dashboard query takes 45 minutes.

Investigation:

1. Open the Spark UI.
2. Navigate to Stages.
3. Find a stage with substantial shuffle activity, such as 10 GB of shuffle write.
4. Compare task durations: median 2 seconds, maximum 40 seconds.
5. Investigate whether one or more partitions contain disproportionately large amounts of data.

This pattern suggests possible skew, but it does not prove skew on its own. Check partition sizes, task input records, shuffle metrics, and the query plan before deciding on a fix.

# cache() vs. persist()

```
Why we need them at all

Spark is lazy: it doesn’t store results. Every action (count, write, show) recomputes the whole chain from the source. If you use the same DataFrame 3 times, Spark reads and transforms it 3 times.

cache() / persist() say: “after computing this once, keep it, and reuse it next time.”

The difference (one line)
cache() = persist() with the default storage level. No options.
persist(level) = you choose where and how to store it: memory, disk, both, serialized, replicated.
```

## Storage Levels

| Storage Level         | What It Means                                                                |
| --------------------- | ---------------------------------------------------------------------------- |
| `MEMORY_ONLY`         | Store data in memory; partitions may be recomputed if evicted                |
| `MEMORY_AND_DISK`     | Store in memory and use disk for partitions that do not fit                  |
| `DISK_ONLY`           | Store persisted data on disk                                                 |
| `MEMORY_ONLY_SER`     | Store serialized data in memory; availability depends on the API             |
| `MEMORY_AND_DISK_SER` | Store serialized data in memory and on disk; availability depends on the API |
| `OFF_HEAP`            | Use off-heap memory when supported and configured                            |

The exact storage-level options depend on whether you use the DataFrame, Dataset, or RDD API and on the Spark version.

## Example: cache()

```python
df = spark.read.parquet("/data/sales")

df.cache()

df.count()
df.groupBy("region").sum().show()
```

Caching is lazy: the data is generally materialized when an action first evaluates it.

## Example: persist()

```python
from pyspark import StorageLevel

df = spark.read.parquet("/data/sales")

df.persist(StorageLevel.MEMORY_AND_DISK)

df.count()
df.groupBy("region").sum().show()
```

Use `unpersist()` when you no longer need the cached data:

```python
df.unpersist()
```

**Practical rule:** Cache or persist a DataFrame when you expect to reuse it multiple times and recomputing it would be expensive. Avoid caching everything, because cached data consumes cluster resources.


# Catalyst Optimizer

Catalyst is Spark’s query optimizer. You write what you want (DataFrame or SQL), and Catalyst figures out the fastest way to do it. It works in 4 phases.

## Stage 1: Analysis (“Does everything exist?”)

Tables, columns.

## Stage 2: Logical optimization (“Rewrite it smarter”)

### What is predicate pushdown and column pruning?

Both mean “read less data from the source”. Catalyst applies them automatically.

**Predicate pushdown:** The filter (WHERE) is sent down to the data source, so rows are skipped while reading instead of after. Parquet/Delta skip whole files or row groups using min/max statistics. JDBC sends the WHERE to the database.

**Column pruning:** Only the columns you use are read. Parquet and Delta are columnar, so the other columns are never touched.

## Stage 3: Physical planning (“Pick HOW to run it”)

Now Spark decides the actual method for each step. The big decision here is the join type:

- Region table is tiny (5 MB), so it picks a Broadcast Hash Join: send the small table to every executor, no shuffle.
- If both tables were huge, it would pick a Sort-Merge Join: shuffle and sort both sides.

## Stage 4: Code generation

Tungsten Engine turns the plan into compact Java bytecode.

Without codegen, Spark runs each step as a separate worker passing rows along a line:

- Worker 1 reads a row and hands it to Worker 2.
- Worker 2 checks the filter and hands it to Worker 3.
- Worker 3 picks the columns and hands it on.

For 1 billion rows, that’s billions of hand-offs. Each hand-off is a small function call, and the overhead adds up.

With codegen, Spark writes one small custom program for your exact query, so a single worker does all 3 steps in one go.

## How do you read `df.explain(True)`?

`explain()` shows Spark’s plan: how it will run your query, without running it.

- `df.explain()` shows only the final plan (the physical plan). This is what you use 90% of the time.
- `df.explain(True)` shows all 4 plans, from your raw code to the final plan, matching Catalyst’s stages.

## Why are built-in functions generally preferable to Python UDFs?

Built-in functions run inside Spark’s engine (the JVM), and Catalyst can optimize them. A Python UDF is a black box Catalyst can’t see inside it, which is slow.

## What is checkpointing, and how does it differ from caching?

Caching keeps data temporarily on the executors for speed and keeps the lineage. Checkpointing saves data permanently to reliable storage (DBFS, S3, HDFS) and cuts the lineage.

# spark-submit

## What it is

`spark-submit` is the command-line tool that launches a Spark application from a code file (`.py` for PySpark, `.jar` for Scala/Java).

## Basic command

```bash
spark-submit \
  --master yarn \
  --deploy-mode cluster \
  --name sales_etl \
  --num-executors 5 \
  --executor-cores 4 \
  --executor-memory 8g \
  --driver-memory 4g \
  --conf spark.sql.shuffle.partitions=100 \
  --py-files utils.zip \
  sales_etl.py --run_date 2026-10-18
```

## The important options

| Option | Meaning |
|---|---|
| `--master` | Where to run: `yarn`, `k8s://...`, `local[*]` (your laptop, all available cores) |
| `--deploy-mode` | `client` or `cluster` |
| `--num-executors` | How many executors |
| `--executor-cores` | CPU cores per executor |
| `--executor-memory` | RAM per executor |
| `--driver-memory` | RAM for the driver |
| `--conf key=value` | Any Spark configuration setting |
| `--py-files` | Extra Python files or ZIP files your code imports |
| `--jars` / `--packages` | Extra libraries, such as a JDBC driver |
| After the file name | Your own arguments, read with `sys.argv` or `argparse` |

**Example:** 5 executors × 4 cores = 20 executor cores. This provides capacity for up to 20 concurrent tasks when all cores are available and tasks are ready to run.

## Client mode vs. cluster mode

| Feature | Client mode | Cluster mode |
|---|---|---|
| Where the driver runs | On the machine where you typed the command | On a node inside the cluster |
| If your terminal closes | The application may stop if the client process terminates | The application can continue running |
| Logs | Available in your terminal | Available through the cluster manager or application logs |
| Common use | Testing and debugging | Production jobs |

## What happens step by step when you run `spark-submit`

1. `spark-submit` contacts the cluster manager, such as YARN or Kubernetes.
2. The cluster manager starts the driver, either on the client machine or inside the cluster, depending on the deploy mode.
3. The driver creates the `SparkSession` and requests executors.
4. Executors start on worker nodes.
5. Your code runs. Actions create jobs, jobs are divided into stages, and stages are divided into tasks that execute on the executors.
6. When the application finishes, the executors are released and the application ends.

## The Python file

```python
import sys
from pyspark.sql import SparkSession

run_date = sys.argv[1]

spark = SparkSession.builder \
    .appName("sales_etl") \
    .getOrCreate()

df = spark.read.parquet(
    f"s3://bank-raw/sales/dt={run_date}"
)

df.groupBy("region") \
  .sum("amount") \
  .write \
  .mode("overwrite") \
  .saveAsTable("gold.sales_by_region")

spark.stop()
```

### How the script receives the argument

The command includes:

```bash
sales_etl.py --run_date 2026-10-18
```

For the Python code shown above, which uses `sys.argv[1]`, pass the date directly as a positional argument:

```bash
spark-submit sales_etl.py 2026-10-18
```

Then:

```python
run_date = sys.argv[1]
```

sets `run_date` to `2026-10-18`.

If you want to use the named argument `--run_date 2026-10-18`, use Python's `argparse` module instead.

## Important difference: Notebook vs. Python script

- **In a Databricks notebook:** The `spark` session is usually provided automatically.
- **In a standalone PySpark script launched with `spark-submit`:** You typically create it

